Load schema definitions and config

In [0]:
%run ../config/config

In [0]:
dbutils.widgets.text("batch_id","")
batch_id=dbutils.widgets.get("batch_id")

In [0]:
silver_table = f"{catalog}.{silver_schema}.carriers"
gold_table = f"{catalog}.{gold_schema}.dim_airlines"

Read silver Delta table into a DataFrame

In [0]:
from pyspark.sql import functions as F

silver_carriers_df=(
    spark.read
    .format("delta")
    .table(silver_table)
    .filter(F.col("batch_id")==batch_id)
)

In [0]:
gold_dim_airlines_df=(
    silver_carriers_df
    .select(
        "carrier_key",
        "airline_name"
    )
)

Write DataFrame to gold dim_airlines Delta table

In [0]:
if not spark.catalog.tableExists(gold_table):

    gold_dim_airlines_df_write=(
        gold_dim_airlines_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(gold_table)
    )

else:
    from delta.tables import DeltaTable

    delta_table=DeltaTable.forName(spark, gold_table)
    (
        delta_table.alias("t")
        .merge(
            silver_carriers_df.alias("s"),
            "t.carrier_key = s.carrier_key"
        )
        .whenMatchedUpdate(
            set={
                "carrier_key": "s.carrier_key",
                "airline_name": "s.airline_name"
            }

        )
        .whenNotMatchedInsertAll()
        .execute()
    )